# 👶 The Little Baby

> A barebones GPT-style LLM implementation — pure Python, zero dependencies.

In [ ]:
########################
# Runtime elements
########################

from src.functions.reload import reload_modules
reload_modules("src")

In [ ]:
########################
# Runtime configuration
########################

runtime_path_inp = input("Enter the runtime path ('same', '<path>'): ").strip().lower()
runtime_plan_inp = input("Enter the runtime plan ('train', 'finetune', 'inference'): ").strip().lower()

In [ ]:
########################
# Runtime variables
########################

if runtime_path_inp == "same":
    runtime_path = "."
else:
    runtime_path = runtime_path_inp
    
runtime_plan = runtime_plan_inp

settings_path = f"{runtime_path}/settings"
configs_path = f"{runtime_path}/configs"
tokenizers_path = f"{runtime_path}/tokenizers"
inputs_path = f"{runtime_path}/inputs"
outputs_path = f"{runtime_path}/outputs"
models_path = f"{runtime_path}/models"

In [ ]:
########################
# Runtime settings
########################

# Set the debug mode
debug = False #Debug Mode [true, false]

# Instantiate the data
input_path = f"{inputs_path}/shakespeare.txt"

# Instantiate the additional settings
train_cache = False #KV Cache [true, false]
infer_cache = False #KV Cache [true, false]

In [ ]:
########################
# Runtime Flow
########################
import uuid
from datetime import datetime as dt
from src.functions.runtime import from_file, to_file
from src.gpt import GPT

match runtime_plan:

    # PLAN TRAIN
    case "train": # Train the model from scratch
        # Generate a UUID (random UUID)
        runtime_uuid = uuid.uuid4()
        # Get the input data
        input_text = from_file(input_path, "plain")
        setting_path = f"{settings_path}/current.json"
        config = from_file(setting_path, "json")
        prompt = None
        # Save the last settings text to a file
        last = { "last_uuid": str(runtime_uuid), "last_prompt": str(prompt) }
        setting_path = f"{settings_path}/last.json"
        to_file(setting_path, "json", last)
        # Instantiate the model
        model = GPT(config)
        # Train the model from scratch
        n_epochs = config["num_epochs"]
        batch_size = config["batch_size"]
        lr = config["lr"]
        model.train(input_text, train_cache, n_epochs, batch_size, lr)
        # Save the trained model parameters
        model_path = f"{models_path}/model_{runtime_uuid}.json"
        json_model = model.params_to_dict()
        to_file(model_path, "json", json_model)
        # Save the tokenizer to a JSON file
        tokenizer_path = f"{tokenizers_path}/tokenizer_{runtime_uuid}.json"
        tokenizer_json = model.tokenizer_to_dict()
        to_file(tokenizer_path, "json", tokenizer_json)
        # Save the report to a JSON file
        report_path = f"{outputs_path}/report_{runtime_uuid}.json"
        json_report = model.report_to_dict()
        to_file(report_path, "json", json_report)
        # Save the runtime settings text to a file
        config_path = f"{configs_path}/config_{runtime_uuid}.json"
        to_file(config_path, "json", config)
        # Inference using the trained model
        model.inference(prompt, infer_cache)
        # Save the completion to a JSON file
        completion_path = f"{outputs_path}/completion_{runtime_uuid}.json"
        json_completion = model.completion_to_dict()
        to_file(completion_path, "json", json_completion)

    # PLAN FINETUNE
    case "finetune": # Finetune a pretrained model
        # Provide the UUID or Last
        runtime_uuid_inp = input("Enter the model configuration ('<uuid>','last'): ").lower()
        # Load the last UUID
        if runtime_uuid_inp == "last":
            setting_path = f"{settings_path}/last.json"   
            setting = from_file(setting_path, "json")
            runtime_uuid = setting['last_uuid']      
        else:
            runtime_uuid = runtime_uuid_inp        
        # Get the input data
        input_text = from_file(input_path, "plain")
        prompt = None
        # Save the last settings text to a file
        last = { "last_uuid": str(runtime_uuid), "last_prompt": str(prompt) }
        setting_path = f"{settings_path}/last.json"
        to_file(setting_path, "json", last)
        # Load the configuration from a file
        config_path = f"{configs_path}/config_{runtime_uuid}.json"
        config = from_file(config_path, "json")
        # Instantiate the model
        model = GPT(config)
        # Load the pre-trained model parameters
        model_path = f"{models_path}/model_{runtime_uuid}.json"
        model_json = from_file(model_path, "json")
        model.params_from_dict(model_json)
        # Load the tokenizer from a JSON file
        tokenizer_path = f"{tokenizers_path}/tokenizer_{runtime_uuid}.json"
        tokenizer_json = from_file(tokenizer_path, "json")
        model.tokenizer.from_dict(tokenizer_json)
        # Fine Tune the model with the new data
        n_epochs = config["num_epochs"]
        batch_size = config["batch_size"]
        lr = config["lr"]
        model.train(input_text, train_cache, n_epochs, batch_size, lr)
        # Save the tokenizer to a JSON file
        tokenizer_path = f"{tokenizers_path}/tokenizer_{runtime_uuid}_finetuned.json"
        tokenizer_json = model.tokenizer_to_dict()
        to_file(tokenizer_path, "json", tokenizer_json)
        # Save the report to a JSON file
        report_path = f"{outputs_path}/report_{runtime_uuid}.json"
        json_report = model.report_to_dict()
        to_file(report_path, "json", json_report)
        # Save the fine-tuned model parameters
        model_path = f"{models_path}/model_{runtime_uuid}_finetuned.json"
        json_model = model.params_to_dict()
        to_file(model_path, "json", json_model)
        # Inference using the finetuned model
        model.inference(prompt, infer_cache)
        # Save the completion to a JSON file
        completion_path = f"{outputs_path}/completion_{runtime_uuid}_finetuned.json"
        json_completion = model.completion_to_dict()
        to_file(completion_path, "json", json_completion)

    #PLAN INFERENCE
    case "inference": # Inference using a pretrained model
        # Provide the UUID or Last
        runtime_uuid_inp = input("Enter the model configuration ('<uuid>','last'): ").lower()
        # Load the last UUID
        if runtime_uuid_inp == "last":
            setting_path = f"{settings_path}/last.json"   
            setting = from_file(setting_path, "json")
            runtime_uuid = setting['last_uuid']
        else:
            runtime_uuid = runtime_uuid_inp
        # Provide the Prompt or Last
        prompt_inpt = input("Enter the prompt for inference ('<prompt>', 'last', 'none'): ").lower()
        # Load the last Prompt
        if prompt_inpt == "last":
            setting_path = f"{settings_path}/last.json"   
            setting = from_file(setting_path, "json")
            prompt = setting['last_prompt']
        elif prompt_inpt == "none":
            prompt = None
        elif prompt_inpt == "":
            prompt = None
        else:
            prompt = prompt_inpt
        # Save the last settings text to a file
        last = { "last_uuid": str(runtime_uuid), "last_prompt": str(prompt) }
        setting_path = f"{settings_path}/last.json"
        to_file(setting_path, "json", last)
        # Load the configuration from a file
        config_path = f"{configs_path}/config_{runtime_uuid}.json"
        config = from_file(config_path, "json")     
         # Instantiate the model
        model = GPT(config)
        # Load the pre-trained or fine-tuned model parameters
        model_path = f"{models_path}/model_{runtime_uuid}.json"
        model_json = from_file(model_path, "json")
        model.params_from_dict(model_json)
        # Inference with the model
        today = dt.today()
        today_ft = today.strftime('%Y%m%d%H%M%S')
        # Load the tokenizer from a JSON file
        tokenizer_path = f"{tokenizers_path}/tokenizer_{runtime_uuid}.json"
        tokenizer_json = from_file(tokenizer_path, "json")
        model.tokenizer.from_dict(tokenizer_json)
        # Inference with the model
        model.inference(prompt, infer_cache)
        # Save the completion to a JSON file
        completion_path = f"{outputs_path}/completion_{runtime_uuid}_{today_ft}.json"
        json_completion = model.completion_to_dict()
        to_file(completion_path, "json", json_completion)